In [ ]:
import os
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# Enforce reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

## 1. Evaluation Framework & Data Split Strategy

**Objective:** To accurately evaluate how well the drone gesture classifier will perform in a real-world deployment (e.g., the upcoming client demo).

**Methodology & Justification:** A standard randomized 80/20 train-validation split would result in *data leakage*. Because the dataset consists of specific subjects (e.g., S1, S3, S11) performing multiple iterations of gestures, randomly splitting instances would mean frames of the same subject appear in both the training and validation sets. In this scenario, the network might memorize subject-specific features (e.g., clothing color, room lighting, body proportions) rather than learning the generalized spatial geometry of the hand gestures. 

To ensure the model generalizes to *unseen humans*—a mandatory requirement for demo day—we implement a **Subject-Wise Split**. We isolate a subset of subjects completely from the training phase and reserve them exclusively for validation. This rigorous evaluation framework guarantees that our validation metrics reflect actual deployment readiness.

In [ ]:
class DroneGestureDataset(Dataset):
    def __init__(self, root_dir, is_train=True, val_subjects=None, transform=None):
        """
        Args:
            root_dir (str): Path to the 'data_resized' directory.
            is_train (bool): If True, loads training subjects. If False, loads validation subjects.
            val_subjects (list): List of subject folder names to be used for validation (e.g., ['S13', 'S14', 'S15']).
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train
        
        if val_subjects is None:
            # Defaulting to 3 subjects for validation to test generalization
            val_subjects = ['S13', 'S14', 'S15'] 
            
        self.instances = [] 
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # Traverse the hierarchical directory structure
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            
            for subject_id in os.listdir(cls_path):
                subj_path = os.path.join(cls_path, subject_id)
                if not os.path.isdir(subj_path): continue

                # Subject-wise split logic
                is_val_subject = subject_id in val_subjects
                if (self.is_train and is_val_subject) or (not self.is_train and not is_val_subject):
                    continue 

                for instance_id in os.listdir(subj_path):
                    inst_path = os.path.join(subj_path, instance_id)
                    if not os.path.isdir(inst_path): continue

                    # Grab all PNG frames and sort them to maintain temporal order (1 to 5)
                    frames = sorted(glob.glob(os.path.join(inst_path, '*.png')))
                    if len(frames) == 5:
                        self.instances.append((frames, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        frame_paths, label = self.instances[idx]
        frames = []

        # Load and transform each frame in the instance sequence
        for path in frame_paths:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        # Stack the 5 frames. Resulting shape: [5, Channels, Height, Width]
        frames_tensor = torch.stack(frames)
        
        return frames_tensor, label

print("Dataset class successfully defined.")